# A/B 测试与灰度发布教程

> **前置知识**: Python基础、基本统计学概念（均值、标准差、假设检验）
>
> **学习目标**: 掌握A/B测试的设计、实施和统计分析方法

---

## 为什么需要 A/B 测试？

```
传统模型上线的问题:
┌─────────────────────────────────────────────────────────────┐
│  "新模型离线指标更好，上线后效果如何？" → 不确定          │
│  "用户会喜欢新功能吗？"                 → 猜测            │
│  "新版本有没有引入bug？"               → 上线才知道       │
└─────────────────────────────────────────────────────────────┘

A/B 测试解决方案:
┌─────────────────────────────────────────────────────────────┐
│  用户请求 → 流量分配 → 模型预测 → 记录结果 → 统计分析     │
│               │                                             │
│        ┌──────┴──────┐                                      │
│        ▼             ▼                                      │
│    控制组(A)     实验组(B)                                   │
│    旧模型        新模型                                      │
│    50%流量       50%流量                                     │
│        │             │                                      │
│        └──────┬──────┘                                      │
│               ▼                                             │
│         统计检验 → 显著性判断 → 上线/回滚决策              │
└─────────────────────────────────────────────────────────────┘
```

## 本教程内容

1. **A/B 测试框架** - 一致性哈希分流、指标收集
2. **统计显著性检验** - 双比例Z检验、p值解读
3. **灰度发布策略** - 渐进式发布、风险控制
4. **自动化发布决策** - 基于数据的自动决策

In [ ]:
# ============================================================
# 环境准备
# ============================================================
# 标准库
import numpy as np
import hashlib      # 用于一致性哈希分流
import random       # 用于模拟随机事件
import time         # 用于时间戳记录

# 科学计算
from scipy import stats  # 统计检验

# 数据结构
from dataclasses import dataclass
from typing import Dict, List, Any
from collections import defaultdict

print("=" * 50)
print("环境准备完成")
print("=" * 50)
print(f"NumPy 版本: {np.__version__}")

## 1. A/B 测试框架

**核心概念**: A/B测试将用户随机分成两组，分别使用不同版本，通过统计方法比较效果差异

```
A/B 测试核心组件:
┌─────────────────────────────────────────────────────────────┐
│                                                             │
│  Variant (变体)                                             │
│  ├── name: 变体名称 (control/treatment)                    │
│  ├── model: 对应的模型                                      │
│  └── weight: 流量权重                                       │
│                                                             │
│  ABTestManager (管理器)                                     │
│  ├── experiments: 实验配置                                  │
│  ├── assignments: 用户分配记录                              │
│  └── metrics: 指标收集                                      │
│                                                             │
└─────────────────────────────────────────────────────────────┘

一致性哈希分流原理:
┌─────────────────────────────────────────────────────────────┐
│  user_id + exp_id → MD5哈希 → 取模得到桶号 → 分配变体      │
│                                                             │
│  优点:                                                      │
│  1. 同一用户每次访问都分到同一组（体验一致）               │
│  2. 不需要存储用户分组信息（无状态）                       │
│  3. 分布均匀，可控制流量比例                               │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# A/B 测试核心类
# ============================================================

@dataclass
class Variant:
    """
    实验变体（版本）
    
    在A/B测试中，每个变体代表一个待测试的版本:
    - control: 控制组，通常是现有的旧版本
    - treatment: 实验组，通常是待验证的新版本
    """
    name: str           # 变体名称，如 "control", "treatment"
    model: Any          # 对应的模型或功能
    weight: float = 1.0 # 流量权重，用于控制分配比例


class ABTestManager:
    """
    A/B 测试管理器
    
    核心功能:
    1. create_experiment(): 创建实验，定义变体
    2. get_variant(): 一致性哈希分流，确保同一用户始终分到同一组
    3. record_metric(): 记录实验指标
    4. get_results(): 获取实验结果统计
    
    一致性哈希原理:
    ┌─────────────────────────────────────────────────────────┐
    │  hash(exp_id + user_id) % 1000 → 桶号(0-999)           │
    │                                                         │
    │  桶号分配:                                              │
    │  [0, 500) → control (50%)                              │
    │  [500, 1000) → treatment (50%)                         │
    │                                                         │
    │  同一用户ID → 同一哈希值 → 同一桶号 → 同一变体        │
    └─────────────────────────────────────────────────────────┘
    """
    
    def __init__(self):
        self.experiments = {}                                    # 实验配置: {exp_id: [variants]}
        self.assignments = {}                                    # 分配记录: {exp_id:user_id: variant}
        self.metrics = defaultdict(lambda: defaultdict(list))   # 指标收集: {exp_id: {metric_key: [values]}}
    
    def create_experiment(self, exp_id: str, variants: List[Variant]):
        """
        创建实验
        
        参数:
            exp_id: 实验唯一标识
            variants: 变体列表，包含控制组和实验组
        """
        self.experiments[exp_id] = variants
        print(f"创建实验 '{exp_id}'，包含 {len(variants)} 个变体:")
        for v in variants:
            print(f"  - {v.name}: 权重={v.weight}")
    
    def get_variant(self, exp_id: str, user_id: str) -> Variant:
        """
        一致性哈希分流
        
        为什么用哈希而不是随机数？
        → 同一用户每次访问都分到同一组，保证体验一致性
        → 不需要存储用户分组信息，无状态设计
        
        参数:
            exp_id: 实验ID
            user_id: 用户ID
            
        返回:
            分配给该用户的变体
        """
        # 生成缓存键
        key = f"{exp_id}:{user_id}"
        
        # 检查是否已分配（缓存）
        if key in self.assignments:
            return self.assignments[key]
        
        # 一致性哈希: MD5 → 整数 → 取模得到桶号
        hash_val = int(hashlib.md5(key.encode()).hexdigest(), 16)
        
        # 根据权重分配变体
        variants = self.experiments[exp_id]
        total_weight = sum(v.weight for v in variants)
        
        # 将哈希值映射到 [0, total_weight) 区间
        threshold = (hash_val % 1000) / 1000 * total_weight
        
        # 累积权重分配
        cumulative = 0
        for variant in variants:
            cumulative += variant.weight
            if threshold < cumulative:
                self.assignments[key] = variant
                return variant
        
        # 兜底返回最后一个变体
        return variants[-1]
    
    def record_metric(self, exp_id: str, variant_name: str, metric: str, value: float):
        """
        记录实验指标
        
        参数:
            exp_id: 实验ID
            variant_name: 变体名称
            metric: 指标名称（如 conversion, revenue）
            value: 指标值
        """
        metric_key = f"{variant_name}_{metric}"
        self.metrics[exp_id][metric_key].append(value)
    
    def get_results(self, exp_id: str) -> Dict:
        """
        获取实验结果统计
        
        返回每个变体的指标统计:
        - mean: 均值
        - std: 标准差
        - count: 样本数
        """
        results = {}
        for key, values in self.metrics[exp_id].items():
            if values:
                results[key] = {
                    'mean': np.mean(values),
                    'std': np.std(values),
                    'count': len(values)
                }
        return results

In [ ]:
# ============================================================
# A/B 测试实战演示
# ============================================================

# 模拟模型类（用于演示）
class MockModel:
    """
    模拟模型
    
    用于演示A/B测试，模拟不同转化率的模型
    """
    def __init__(self, conversion_rate: float):
        """
        参数:
            conversion_rate: 转化率，范围 [0, 1]
                            例如 0.10 表示 10% 的转化率
        """
        self.conversion_rate = conversion_rate
    
    def predict(self, x):
        """模拟预测，根据转化率随机返回是否转化"""
        return random.random() < self.conversion_rate


# ============================================================
# 创建 A/B 测试实验
# ============================================================
print("=" * 60)
print("A/B 测试实验演示")
print("=" * 60)

ab_manager = ABTestManager()

# 创建实验：对比两个模型
# - control: 旧模型，10% 转化率
# - treatment: 新模型，12% 转化率（提升 20%）
ab_manager.create_experiment('model_test', [
    Variant('control', MockModel(0.10), weight=1.0),    # 控制组：50% 流量
    Variant('treatment', MockModel(0.12), weight=1.0)   # 实验组：50% 流量
])

# ============================================================
# 模拟用户流量
# ============================================================
print("\n" + "-" * 60)
print("模拟 1000 个用户请求...")
print("-" * 60)

# 统计分配情况
assignment_counts = {'control': 0, 'treatment': 0}

for i in range(1000):
    user_id = f"user_{i}"
    
    # 1. 获取用户分配的变体
    variant = ab_manager.get_variant('model_test', user_id)
    assignment_counts[variant.name] += 1
    
    # 2. 使用对应模型进行预测
    converted = variant.model.predict(None)
    
    # 3. 记录转化指标
    ab_manager.record_metric('model_test', variant.name, 'conversion', int(converted))

# ============================================================
# 查看实验结果
# ============================================================
print("\n" + "=" * 60)
print("实验结果")
print("=" * 60)

# 流量分配情况
print("\n流量分配:")
for name, count in assignment_counts.items():
    print(f"  {name}: {count} 用户 ({count/10:.1f}%)")

# 转化率统计
print("\n转化率统计:")
results = ab_manager.get_results('model_test')
for key, stats in results.items():
    print(f"  {key}:")
    print(f"    转化率: {stats['mean']:.2%}")
    print(f"    样本数: {stats['count']}")

## 2. 统计显著性检验

**核心问题**: 实验组比控制组好，是真的好还是随机波动？

```
统计显著性检验原理:
┌─────────────────────────────────────────────────────────────┐
│                                                             │
│  原假设 H0: 两组没有差异 (p1 = p2)                         │
│  备择假设 H1: 两组有差异 (p1 ≠ p2)                         │
│                                                             │
│  检验流程:                                                  │
│  1. 计算两组转化率 p1, p2                                  │
│  2. 计算合并转化率 p_pool                                  │
│  3. 计算标准误 SE                                          │
│  4. 计算 Z 统计量 = (p2 - p1) / SE                         │
│  5. 计算 p 值（双尾检验）                                  │
│  6. 如果 p < α (通常0.05)，拒绝原假设，认为有显著差异     │
│                                                             │
└─────────────────────────────────────────────────────────────┘

p 值解读:
┌─────────────────────────────────────────────────────────────┐
│  p < 0.01  : 非常显著，差异几乎不可能是随机的              │
│  p < 0.05  : 显著，差异很可能不是随机的（常用阈值）        │
│  p < 0.10  : 边缘显著，需要更多数据                        │
│  p >= 0.10 : 不显著，无法排除随机波动                      │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# 统计显著性检验函数
# ============================================================

def calculate_significance(
    control_conversions: int,
    control_total: int,
    treatment_conversions: int,
    treatment_total: int,
    alpha: float = 0.05
) -> Dict:
    """
    双比例 Z 检验
    
    用于判断两组转化率的差异是否具有统计显著性
    
    数学原理:
    ┌─────────────────────────────────────────────────────────┐
    │  p1 = control_conversions / control_total   (控制组率) │
    │  p2 = treatment_conversions / treatment_total (实验组率)│
    │                                                         │
    │  合并转化率:                                            │
    │  p_pool = (c1 + c2) / (n1 + n2)                        │
    │                                                         │
    │  标准误:                                                │
    │  SE = sqrt(p_pool * (1-p_pool) * (1/n1 + 1/n2))       │
    │                                                         │
    │  Z 统计量:                                              │
    │  Z = (p2 - p1) / SE                                    │
    │                                                         │
    │  p 值 (双尾):                                           │
    │  p_value = 2 * (1 - Φ(|Z|))                            │
    └─────────────────────────────────────────────────────────┘
    
    参数:
        control_conversions: 控制组转化数
        control_total: 控制组总数
        treatment_conversions: 实验组转化数
        treatment_total: 实验组总数
        alpha: 显著性水平，默认 0.05
        
    返回:
        包含检验结果的字典
    """
    # 计算转化率
    p1 = control_conversions / control_total      # 控制组转化率
    p2 = treatment_conversions / treatment_total  # 实验组转化率
    
    # 计算合并转化率（假设两组没有差异时的最佳估计）
    p_pool = (control_conversions + treatment_conversions) / (control_total + treatment_total)
    
    # 计算标准误
    se = np.sqrt(p_pool * (1 - p_pool) * (1/control_total + 1/treatment_total))
    
    # 处理标准误为0的情况（两组都是0%或100%转化）
    if se == 0:
        return {'significant': False, 'p_value': 1.0, 'reason': '标准误为0，无法计算'}
    
    # 计算 Z 统计量
    z = (p2 - p1) / se
    
    # 计算 p 值（双尾检验）
    # 双尾检验：检测"是否有差异"，不管是变好还是变差
    p_value = 2 * (1 - stats.norm.cdf(abs(z)))
    
    # 计算相对提升（lift）
    lift = (p2 - p1) / p1 * 100 if p1 > 0 else 0
    
    return {
        'control_rate': p1,           # 控制组转化率
        'treatment_rate': p2,         # 实验组转化率
        'lift': lift,                 # 相对提升百分比
        'z_score': z,                 # Z 统计量
        'p_value': p_value,           # p 值
        'significant': p_value < alpha,  # 是否显著
        'recommendation': '采用新版本' if (p_value < alpha and lift > 0) else 
                         ('回滚旧版本' if (p_value < alpha and lift < 0) else '继续观察')
    }


# ============================================================
# 显著性检验演示
# ============================================================
print("=" * 60)
print("统计显著性检验演示")
print("=" * 60)

# 场景1：明显的提升
print("\n场景1: 明显的提升")
print("-" * 40)
result1 = calculate_significance(
    control_conversions=100, control_total=1000,    # 控制组: 10% 转化率
    treatment_conversions=150, treatment_total=1000  # 实验组: 15% 转化率
)
for k, v in result1.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")
    else:
        print(f"  {k}: {v}")

# 场景2：微小的差异（可能不显著）
print("\n场景2: 微小的差异")
print("-" * 40)
result2 = calculate_significance(
    control_conversions=100, control_total=1000,    # 控制组: 10% 转化率
    treatment_conversions=105, treatment_total=1000  # 实验组: 10.5% 转化率
)
for k, v in result2.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")
    else:
        print(f"  {k}: {v}")

# 场景3：样本量不足
print("\n场景3: 样本量不足")
print("-" * 40)
result3 = calculate_significance(
    control_conversions=10, control_total=100,    # 控制组: 10% 转化率
    treatment_conversions=15, treatment_total=100  # 实验组: 15% 转化率
)
for k, v in result3.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")
    else:
        print(f"  {k}: {v}")

print("\n" + "=" * 60)
print("结论: 相同的提升幅度，样本量越大越容易达到显著性")
print("=" * 60)

## 3. 灰度发布策略

**核心概念**: 灰度发布（渐进式发布）是逐步将新版本推送给更多用户的策略，降低风险

```
灰度发布流程:
┌─────────────────────────────────────────────────────────────┐
│                                                             │
│  阶段1: 金丝雀发布 (1-5%)                                   │
│  ├── 只给少量用户使用新版本                                │
│  ├── 密切监控错误率和性能                                  │
│  └── 发现问题立即回滚                                      │
│                                                             │
│  阶段2: 小规模验证 (5-20%)                                  │
│  ├── 扩大测试范围                                          │
│  ├── 收集更多数据验证效果                                  │
│  └── 确认无重大问题                                        │
│                                                             │
│  阶段3: 大规模推广 (20-50%)                                 │
│  ├── 进一步扩大范围                                        │
│  ├── 进行统计显著性检验                                    │
│  └── 确认效果符合预期                                      │
│                                                             │
│  阶段4: 全量发布 (100%)                                     │
│  ├── 完全切换到新版本                                      │
│  └── 保留旧版本用于紧急回滚                                │
│                                                             │
└─────────────────────────────────────────────────────────────┘

发布策略对比:
┌─────────────────────────────────────────────────────────────┐
│  策略        风险    回滚速度    适用场景                   │
│  ────────    ────    ────────    ────────                   │
│  灰度发布    低      快          大多数场景                 │
│  蓝绿部署    中      极快        需要快速切换               │
│  金丝雀      低      快          高风险变更                 │
│  影子模式    无      N/A         无风险验证                 │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# 灰度发布管理器
# ============================================================

class GradualRollout:
    """
    渐进式发布管理器
    
    核心功能:
    1. start(): 开始灰度发布，设置初始比例
    2. get_model(): 根据用户ID分配模型版本
    3. increase(): 增加新版本流量比例
    4. rollback(): 紧急回滚到旧版本
    5. complete(): 完成发布，全量切换
    
    流量分配原理:
    ┌─────────────────────────────────────────────────────────┐
    │  hash(user_id) % 100 → 桶号(0-99)                      │
    │                                                         │
    │  rollout_pct = 30% 时:                                 │
    │  [0, 30) → 新模型 (30%)                                │
    │  [30, 100) → 旧模型 (70%)                              │
    └─────────────────────────────────────────────────────────┘
    """
    
    def __init__(self, old_model, new_model):
        """
        参数:
            old_model: 旧版本模型（当前生产环境）
            new_model: 新版本模型（待验证）
        """
        self.old_model = old_model
        self.new_model = new_model
        self.rollout_pct = 0  # 新版本流量比例，初始为0
        self.metrics = {'old': [], 'new': []}  # 分别记录两个版本的指标
    
    def start(self, initial_pct: int = 5):
        """
        开始灰度发布
        
        参数:
            initial_pct: 初始流量比例，建议从小比例开始（1-5%）
        """
        self.rollout_pct = initial_pct
        print(f"开始灰度发布，新版本流量: {initial_pct}%")
    
    def get_model(self, user_id: str):
        """
        根据用户ID分配模型版本
        
        使用哈希确保同一用户始终使用同一版本
        
        返回:
            (model, model_type): 模型实例和类型标识
        """
        # 哈希分桶
        hash_val = hash(user_id) % 100
        
        # 根据发布比例分配
        if hash_val < self.rollout_pct:
            return self.new_model, 'new'
        return self.old_model, 'old'
    
    def record(self, model_type: str, success: bool):
        """记录一次请求的结果"""
        self.metrics[model_type].append(int(success))
    
    def increase(self, increment: int = 5):
        """
        增加新版本流量比例
        
        参数:
            increment: 增加的百分比，建议每次增加5-10%
        """
        old_pct = self.rollout_pct
        self.rollout_pct = min(100, self.rollout_pct + increment)
        print(f"流量比例: {old_pct}% → {self.rollout_pct}%")
    
    def rollback(self):
        """紧急回滚：将新版本流量降为0"""
        self.rollout_pct = 0
        print("⚠️ 紧急回滚！新版本流量已降为 0%")
    
    def complete(self):
        """完成发布：全量切换到新版本"""
        self.rollout_pct = 100
        print("✓ 发布完成！新版本流量已达 100%")
    
    def get_stats(self) -> Dict:
        """
        获取两个版本的统计信息
        
        返回:
            各版本的成功率和样本数
        """
        stats = {}
        for model_type in ['old', 'new']:
            data = self.metrics[model_type]
            stats[model_type] = {
                'success_rate': np.mean(data) if data else 0,
                'count': len(data)
            }
        return stats

In [ ]:
# ============================================================
# 灰度发布实战演示
# ============================================================
print("=" * 60)
print("灰度发布实战演示")
print("=" * 60)

# 创建灰度发布管理器
# - 旧模型: 10% 转化率
# - 新模型: 12% 转化率（提升 20%）
rollout = GradualRollout(
    old_model=MockModel(0.10),
    new_model=MockModel(0.12)
)

# ============================================================
# 阶段1: 金丝雀发布 (10%)
# ============================================================
print("\n" + "-" * 60)
print("阶段1: 金丝雀发布")
print("-" * 60)
rollout.start(10)

# 模拟 500 个请求
for i in range(500):
    model, model_type = rollout.get_model(f"user_{i}")
    success = model.predict(None)
    rollout.record(model_type, success)

stats = rollout.get_stats()
print(f"\n统计结果:")
print(f"  旧版本: 成功率={stats['old']['success_rate']:.2%}, 样本数={stats['old']['count']}")
print(f"  新版本: 成功率={stats['new']['success_rate']:.2%}, 样本数={stats['new']['count']}")

# ============================================================
# 阶段2: 扩大范围 (30%)
# ============================================================
print("\n" + "-" * 60)
print("阶段2: 扩大范围")
print("-" * 60)
rollout.increase(20)  # 10% → 30%

# 模拟更多请求
for i in range(500, 1500):
    model, model_type = rollout.get_model(f"user_{i}")
    success = model.predict(None)
    rollout.record(model_type, success)

stats = rollout.get_stats()
print(f"\n统计结果:")
print(f"  旧版本: 成功率={stats['old']['success_rate']:.2%}, 样本数={stats['old']['count']}")
print(f"  新版本: 成功率={stats['new']['success_rate']:.2%}, 样本数={stats['new']['count']}")

# ============================================================
# 阶段3: 统计检验决策
# ============================================================
print("\n" + "-" * 60)
print("阶段3: 统计检验决策")
print("-" * 60)

# 进行显著性检验
old_data = rollout.metrics['old']
new_data = rollout.metrics['new']

result = calculate_significance(
    control_conversions=sum(old_data),
    control_total=len(old_data),
    treatment_conversions=sum(new_data),
    treatment_total=len(new_data)
)

print(f"\n显著性检验结果:")
print(f"  旧版本转化率: {result['control_rate']:.2%}")
print(f"  新版本转化率: {result['treatment_rate']:.2%}")
print(f"  相对提升: {result['lift']:.1f}%")
print(f"  p 值: {result['p_value']:.4f}")
print(f"  是否显著: {result['significant']}")
print(f"  建议: {result['recommendation']}")

# ============================================================
# 阶段4: 根据结果决策
# ============================================================
print("\n" + "-" * 60)
print("阶段4: 最终决策")
print("-" * 60)

if result['significant'] and result['lift'] > 0:
    rollout.complete()
    print("新版本效果显著提升，完成全量发布！")
elif result['significant'] and result['lift'] < 0:
    rollout.rollback()
    print("新版本效果显著下降，紧急回滚！")
else:
    print(f"当前流量比例: {rollout.rollout_pct}%")
    print("效果不显著，建议继续观察或增加样本量")

## 4. 自动化发布决策

**核心概念**: 基于统计检验结果自动决定是否继续发布、回滚或等待

```
自动化决策流程:
┌─────────────────────────────────────────────────────────────┐
│                                                             │
│  收集数据 → 样本量检查 → 显著性检验 → 自动决策            │
│                                                             │
│  决策逻辑:                                                  │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  样本量 < 最小要求?                                 │   │
│  │  └── 是 → 等待，继续收集数据                       │   │
│  │                                                     │   │
│  │  p 值 < α (显著)?                                   │   │
│  │  ├── 是 + 提升 > 0 → 继续发布 / 全量上线           │   │
│  │  ├── 是 + 提升 < 0 → 紧急回滚                      │   │
│  │  └── 否 → 等待，继续收集数据                       │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
└─────────────────────────────────────────────────────────────┘

安全阈值设置:
┌─────────────────────────────────────────────────────────────┐
│  参数              推荐值      说明                         │
│  ──────────        ──────      ──────                       │
│  min_samples       100-500     最小样本量，确保统计可靠     │
│  significance      0.05        显著性水平，越小越严格       │
│  min_lift          0%          最小提升要求                 │
│  max_degradation   -5%         最大允许下降，超过则回滚     │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# 自动化发布决策器
# ============================================================

class AutomatedRollout(GradualRollout):
    """
    自动化发布决策器
    
    继承自 GradualRollout，增加自动决策功能
    
    决策逻辑:
    ┌─────────────────────────────────────────────────────────┐
    │  1. 检查样本量是否足够                                 │
    │  2. 进行统计显著性检验                                 │
    │  3. 根据结果自动决策:                                  │
    │     - proceed: 继续发布（显著提升）                    │
    │     - rollback: 紧急回滚（显著下降）                   │
    │     - wait: 继续观察（不显著）                         │
    └─────────────────────────────────────────────────────────┘
    """
    
    def __init__(
        self,
        old_model,
        new_model,
        min_samples: int = 100,
        significance_level: float = 0.05
    ):
        """
        参数:
            old_model: 旧版本模型
            new_model: 新版本模型
            min_samples: 每组最小样本量，确保统计可靠性
            significance_level: 显著性水平，默认 0.05
        """
        super().__init__(old_model, new_model)
        self.min_samples = min_samples
        self.significance_level = significance_level
    
    def should_proceed(self) -> tuple:
        """
        自动决策是否继续发布
        
        返回:
            (decision, reason): 决策和原因
            - decision: 'proceed' / 'rollback' / 'wait'
            - reason: 决策原因说明
        """
        old_data = self.metrics['old']
        new_data = self.metrics['new']
        
        # 检查样本量
        if len(old_data) < self.min_samples or len(new_data) < self.min_samples:
            return 'wait', f"样本量不足 (旧:{len(old_data)}, 新:{len(new_data)}, 需要:{self.min_samples})"
        
        # 进行显著性检验
        result = calculate_significance(
            control_conversions=sum(old_data),
            control_total=len(old_data),
            treatment_conversions=sum(new_data),
            treatment_total=len(new_data),
            alpha=self.significance_level
        )
        
        # 根据结果决策
        if result['significant']:
            if result['lift'] > 0:
                return 'proceed', f"显著提升 {result['lift']:.1f}% (p={result['p_value']:.4f})"
            else:
                return 'rollback', f"显著下降 {result['lift']:.1f}% (p={result['p_value']:.4f})"
        
        return 'wait', f"效果不显著 (p={result['p_value']:.4f})"


# ============================================================
# 自动化发布演示
# ============================================================
print("=" * 60)
print("自动化发布决策演示")
print("=" * 60)

# 场景: 新模型有明显提升 (10% → 15%)
auto_rollout = AutomatedRollout(
    old_model=MockModel(0.10),
    new_model=MockModel(0.15),  # 50% 相对提升
    min_samples=200,
    significance_level=0.05
)

# 开始灰度发布
auto_rollout.start(50)

# 模拟流量并检查决策
print("\n" + "-" * 60)
print("模拟流量并检查自动决策")
print("-" * 60)

for batch in range(5):
    # 每批 200 个请求
    for i in range(200):
        user_id = f"user_{batch * 200 + i}"
        model, model_type = auto_rollout.get_model(user_id)
        success = model.predict(None)
        auto_rollout.record(model_type, success)
    
    # 检查决策
    decision, reason = auto_rollout.should_proceed()
    stats = auto_rollout.get_stats()
    
    print(f"\n批次 {batch + 1} (累计 {(batch + 1) * 200} 请求):")
    print(f"  旧版本: {stats['old']['success_rate']:.2%} ({stats['old']['count']} 样本)")
    print(f"  新版本: {stats['new']['success_rate']:.2%} ({stats['new']['count']} 样本)")
    print(f"  决策: {decision}")
    print(f"  原因: {reason}")
    
    # 根据决策执行操作
    if decision == 'proceed':
        if auto_rollout.rollout_pct < 100:
            auto_rollout.increase(25)
        else:
            print("\n✓ 已完成全量发布！")
            break
    elif decision == 'rollback':
        auto_rollout.rollback()
        print("\n⚠️ 已执行紧急回滚！")
        break

## 总结

本教程介绍了 A/B 测试与灰度发布的核心概念和实现方法：

### 核心组件回顾

| 组件 | 功能 | 关键方法 |
|:-----|:-----|:---------|
| ABTestManager | A/B 测试管理 | get_variant(), record_metric() |
| calculate_significance | 统计显著性检验 | 双比例 Z 检验 |
| GradualRollout | 灰度发布 | start(), increase(), rollback() |
| AutomatedRollout | 自动化决策 | should_proceed() |

### 发布策略对比

| 策略 | 风险 | 回滚速度 | 适用场景 |
|:-----|:-----|:---------|:---------|
| A/B 测试 | 低 | 快 | 功能对比、模型验证 |
| 金丝雀发布 | 低 | 快 | 高风险变更、新版本验证 |
| 蓝绿部署 | 中 | 极快 | 需要快速切换的场景 |
| 影子模式 | 无 | N/A | 无风险测试、性能验证 |

### 最佳实践

```
A/B 测试检查清单:
✓ 使用一致性哈希确保用户体验一致
✓ 设置足够的最小样本量（通常 100-500）
✓ 使用统计显著性检验（p < 0.05）
✓ 从小流量开始，逐步扩大
✓ 设置自动回滚机制
✓ 记录实验结果用于复盘

常见陷阱:
✗ 样本量不足就下结论
✗ 忽略统计显著性，只看绝对数值
✗ 同时运行太多实验导致流量稀释
✗ 没有设置回滚机制
```

### 下一步学习

- **06_AutoRetraining_tutorial.ipynb**: 自动重训练策略
- **07_FeatureStore_tutorial.ipynb**: 特征存储与管理